In [1]:
# import features
import pandas as pd
df_pulse = pd.read_csv('output/processed_data/pulse_mean.csv')
df_entropy = pd.read_csv('output/processed_data/entropy_rates.csv')
df_mm = pd.read_csv('output/processed_data/muscle_mass_mean.csv')
df_bw = pd.read_csv('output/processed_data/BodyWeight_mean.csv')
df_tbw = pd.read_csv('output/processed_data/TotalBodyWater_mean.csv')
df_dbp = pd.read_csv('output/processed_data/DiastolicBloodPressure_mean.csv')
df_sbp = pd.read_csv('output/processed_data/SystolicBloodPressure_mean.csv')
df_bt = pd.read_csv('output/processed_data/BodyTemperature_mean.csv')

# marker/ elements of delirium
df_agitation = pd.read_csv('output/processed_data/agitation_label.csv')
df_sleepAnomaly = pd.read_csv('output/processed_data/sleep_duration_anomalies.csv')

merged_df = df_pulse
print(merged_df.shape)
# Merge with the remaining dataframes
for df in [ df_entropy, df_mm, df_bw, df_tbw, df_dbp, df_sbp, df_bt, df_agitation, df_sleepAnomaly]:
    print(df.shape)
    merged_df = pd.merge(merged_df, df, on=['patient_id', 'date'], how='outer')
    
merged_df

In [29]:
df_person = merged_df[merged_df['patient_id'] == "55cd4"].copy()
df_person['agitation'] = df_person['agitation'].fillna(0)
df_person['sleepAnomaly'] = df_person['sleepAnomaly'].fillna(0)
df_person_dropna = df_person.dropna(axis=0, how='any', inplace = False)
df_person_dropna

In [30]:
# sleepAnomaly = df_person_dropna[df_person_dropna.sleepAnomaly == 1].date.values
# sleepAnomaly
sleepAnomaly_date = pd.to_datetime(df_person_dropna[df_person_dropna.sleepAnomaly == 1].date).dt.strftime('%m-%d').astype(str).values
sleepAnomaly_date

In [31]:
# agitation_date = df_person_dropna[df_person_dropna.agitation == 1].date.values
# agitation_date

agitation_date = pd.to_datetime(df_person_dropna[df_person_dropna.agitation == 1].date).dt.strftime('%m-%d').astype(str).values
agitation_date

In [32]:
from sklearn.preprocessing import StandardScaler
data = df_person_dropna.copy()
features = data[['heart_rate', 'entropy_rate', 'muscle_mass', 'Body_Weight', 'Total_Body_Water', 'Diastolic_blood_pressure', 'Systolic_blood_pressure', 'body_temperature']]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
scaled_features_df = pd.DataFrame(scaled_features, columns=features.columns)
scaled_features_df

In [33]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from ucimlrepo import fetch_ucirepo

# Parameters
n_estimators = 30  # Number of trees
contamination = 0.1  # Expected proportion of anomalies
sample_size = 14  # Number of samples used to train each tree

# Train Isolation Forest
iso_forest = IsolationForest(n_estimators=n_estimators,
                            contamination=contamination,
                            max_samples=sample_size,
                            random_state=42)

iso_forest.fit(scaled_features_df)

# Calculate anomaly scores and classify anomalies

data['anomaly_score'] = iso_forest.decision_function(scaled_features_df)
data['anomaly'] = iso_forest.predict(scaled_features_df)

data['anomaly'].value_counts()

In [34]:
anomaly_dates = pd.to_datetime(data[data.anomaly == -1].date).dt.strftime('%m-%d').astype(str).values
anomaly_dates

In [35]:
data_plot = data[['date', 'heart_rate', 'entropy_rate', 'muscle_mass', 'Body_Weight', 'Total_Body_Water', 'Diastolic_blood_pressure', 'Systolic_blood_pressure', 'body_temperature']].copy()

data_plot['date'] = pd.to_datetime(data_plot['date'])            
data_plot['date'] = data_plot['date'].dt.strftime('%m-%d')
data_plot['date'] = data_plot['date'].astype(str)             
data_plot.set_index('date', inplace=True)

data_plot

In [37]:
# Set the figure size
plt.figure(figsize=(12, 10))

# Plot each variable
for i, column in enumerate(data_plot.columns):
    plt.subplot(3, 3, i+1)  # 3x3 grid for 7 variables, including empty ones for better arrangement
    plt.plot(data_plot.index, data_plot[column], label=column)
    
    # # Highlight specific index in red
    plt.scatter(anomaly_dates, data_plot[column][anomaly_dates], color='red', label='anomaly', zorder=5)
    for l in sleepAnomaly_date:
        plt.axvline(l, color='purple', linestyle='--', label='sleep')
    # for l2 in agitation_date:
    #     plt.axvline(l2, color='blue', linestyle='--', label='Agitation')    
    
    # Title and labels
    plt.title(f'{column} ')
    plt.xlabel('')
    plt.ylabel('')
    plt.xticks(rotation=90)
    
    # Optional: Adjusting the y-axis range to fit all data in the plots
    plt.ylim(data_plot[column].min() - 1, data_plot[column].max() + 1)

# Adjust layout and show plot
plt.tight_layout()

plt.savefig('output/forest_isolation/anomaly_before_sleepChange_55cd4.png', dpi=300)
plt.show()